# EAGLE-3 draft model 학습과 vLLM 실행

Speculative decoding에서는 draft model이 여러 토큰을 제안하고 target model이 검증한다. 이 실습은 **Qwen/Qwen3.8-27B**를 고정한 채 작은 EAGLE-3 모델을 학습한다. target model의 전체 가중치를 다시 학습하지 않는다.

| 방법 | 사용 전에 직접 학습해야 하는가? |
| --- | --- |
| N-gram, Suffix | 별도 모델을 학습하지 않는다. 입력이나 이전 출력의 토큰 패턴을 사용한다. |
| 일반 draft model | 호환되는 기존 소형 언어 모델을 사용할 수 있다. 추가 학습은 선택 사항이다. |
| EAGLE-3, DFlash | target에 맞는 draft 가중치가 필요하다. 공개 가중치를 사용하거나 직접 학습한다. |
| Native MTP | target에 포함된 학습된 MTP 가중치와 엔진 지원이 있으면 별도 학습 없이 사용한다. |

[vLLM 안내](https://docs.vllm.ai/en/stable/features/speculative_decoding/)가 연결하는 [Speculators 학습 가이드](https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train/)의 오프라인 EAGLE-3 절차를 사용한다. EAGLE-3는 target의 일부 층에서 얻은 hidden state와 토큰 임베딩으로 다음 토큰을 예측하도록 학습한다. 일반 소형 LLM을 처음부터 사전학습하는 과정과 다르다.

전체 셀을 한 번에 실행하지 않는다. 서버 시작·종료 안내마다 별도 터미널에서 조작한다. 실제 GPU 학습과 성능 수치는 아직 검증하지 않았다.

## 1. 환경과 설정

Linux GPU 머신에서 README의 학습 가상환경을 준비하고 그 Python을 노트북 커널로 선택한다. vLLM은 별도 환경에서 실행한다. `VLLM_PYTHON`에는 기존 vLLM 환경의 Python 절대 경로를 지정한다.

`HARDWARE="h100"`은 한 서버의 H100 4개, `"b300"`은 B300 1개를 사용한다. 각 머신에서 이 폴더를 작업 디렉터리로 연다. 서버와 노트북은 같은 머신에서 실행하며 디스크 경로도 공유한다. GPU 번호는 할당받은 번호로 수정한다.

기본값은 512개 응답, 최대 1,024토큰, 3 epoch이다. 학습 절차를 확인하기 위한 크기이며, 실제 가속을 얻으려면 더 많은 대표 데이터와 평가가 필요하다.

In [ ]:
import json
import os
import shlex
import statistics
import subprocess
import sys
from pathlib import Path
from time import perf_counter
from urllib.request import Request, urlopen

from datasets import load_dataset
from transformers import AutoConfig, AutoTokenizer

MODEL = "Qwen/Qwen3.8-27B"
HARDWARE = "h100"  # "h100" 또는 "b300"
GPUS, TP = {"h100": ("0,1,2,3", 4), "b300": ("0", 1)}[HARDWARE]
VLLM_PYTHON = "/path/to/vllm/bin/python"
BASE_URL = "http://127.0.0.1:8000"
OUTPUT = Path("output").resolve()
NUM_SAMPLES = 512
SEQ_LENGTH = 1024
MAX_NEW_TOKENS = 256
EPOCHS = 3
NUM_EVAL = 16

tokenizer = AutoTokenizer.from_pretrained(MODEL)
text_config = AutoConfig.from_pretrained(MODEL).text_config
n_layers = text_config.num_hidden_layers
target_layers = [2, n_layers // 2, n_layers - 3]
OUTPUT.mkdir(exist_ok=True)

VLLM = [VLLM_PYTHON, "-m", "vllm.entrypoints.cli.main"]
SPECULATORS = [sys.executable, "-m", "speculators"]
base_command = [
    "env", f"CUDA_VISIBLE_DEVICES={GPUS}",
    *VLLM, "serve", MODEL,
    "--host", "127.0.0.1", "--port", "8000",
    "--tensor-parallel-size", str(TP),
    "--dtype", "bfloat16",
    "--language-model-only",
    "--max-model-len", "2048",
    "--max-num-seqs", "8",
    "--max-num-batched-tokens", "2048",
    "--disable-prefix-caching",
    "--generation-config", "vllm",
]

### 저장 항목과 공간

- `output/target_responses.jsonl`: target이 생성한 토큰과 학습할 위치의 마스크.
- `output/prepared/`: Speculators가 읽는 전처리 데이터, 길이, 토큰 빈도와 어휘 대응표. 라이브러리의 데이터 형식 정보도 포함된다.
- `output/hidden_states/`: 학습용 hidden state. `output/transfer/`는 서버가 추출한 텐서를 전달하는 작업 경로다.
- `output/checkpoints/`: draft 가중치, 모델 구조·target 참조·어휘 대응 설정, optimizer·scheduler 상태와 검증 손실. 모델을 다시 불러오거나 학습을 이어가는 데 사용한다.

노트북 메타데이터는 Python 커널 정보만 사용한다. 별도 실행 식별자나 체크섬을 작성하지 않는다. 학습 실행 파일은 Speculators의 실행 이력 저장 함수를 비활성화한다. 학습 loss는 셀 출력으로 확인하며 외부 실험 추적기를 사용하지 않는다.

아래 계산은 BF16 hidden state 텐서만의 예상 상한이다. 실제 길이, 파일 부가 정보, 다운로드 캐시와 체크포인트는 별도다. 모델·데이터·길이를 바꾼 실험은 다른 빈 출력 폴더에서 수행한다. 기존 폴더에는 라이브러리의 캐시 재사용·학습 재개 동작이 적용될 수 있다.

In [ ]:
hidden_gib = (
    NUM_SAMPLES * SEQ_LENGTH * (len(target_layers) + 1)
    * text_config.hidden_size * 2 / 1024**3
)
print(f"hidden state 텐서 예상 상한: {hidden_gib:.1f} GiB")
print("추출할 층:", target_layers + [n_layers])

## 2. 일반 vLLM 서버 시작

아래 셀이 출력한 명령을 같은 머신의 별도 터미널에서 실행한다. 서버 시작 완료 메시지 후 다음 셀을 실행한다. 이 서버로 학습 응답을 생성하고 EAGLE-3를 적용하기 전 성능을 측정한다.

Qwen3.8-27B의 언어 모델 BF16 가중치는 약 54GB이고 실행 메모리는 추가로 필요하다. H100에서는 TP 4, B300에서는 TP 1을 사용한다. 텍스트만 처리하며 양자화는 적용하지 않는다.

In [ ]:
print(shlex.join(base_command))

## 3. Target의 응답으로 학습 데이터 생성

UltraChat의 `train_sft`에서 첫 사용자 질문만 가져온다. 기존 assistant 답변은 학습에 사용하지 않고 Qwen3.8-27B가 다시 생성한다. `enable_thinking=False`로 템플릿을 적용하고 greedy decoding을 사용한다.

문자열을 다시 토큰화하면 실제 생성 토큰과 달라질 수 있으므로, vLLM의 `return_token_ids`로 받은 토큰을 저장한다. `loss_mask`는 프롬프트에서 0, target이 생성한 부분에서 1이다. 생성 길이를 포함해 1,024토큰을 넘는 질문은 제외한다. 실제로는 EOS에 의해 256토큰보다 먼저 종료될 수 있다.

데이터 파일은 새로 생성한다. 파일이 이미 있으면 해당 셀을 다시 실행하지 말고 다음 단계부터 진행한다.

In [ ]:
def complete(prompt_ids):
    request = Request(
        f"{BASE_URL}/v1/completions",
        data=json.dumps({
            "model": MODEL,
            "prompt": prompt_ids,
            "max_tokens": MAX_NEW_TOKENS,
            "temperature": 0,
            "return_token_ids": True,
        }).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urlopen(request) as response:
        return json.load(response)["choices"][0]["token_ids"]

train_data = load_dataset(
    "HuggingFaceH4/ultrachat_200k", split="train_sft", streaming=True,
)
count = 0
with (OUTPUT / "target_responses.jsonl").open("x", encoding="utf-8") as file:
    for row in train_data:
        prompt_ids = tokenizer.apply_chat_template(
            [row["messages"][0]], add_generation_prompt=True,
            enable_thinking=False, return_dict=False,
        )
        if len(prompt_ids) + MAX_NEW_TOKENS > SEQ_LENGTH:
            continue
        generated_ids = complete(prompt_ids)
        file.write(json.dumps({
            "input_ids": prompt_ids + generated_ids,
            "loss_mask": [0] * len(prompt_ids) + [1] * len(generated_ids),
        }) + "\n")
        count += 1
        if count == NUM_SAMPLES:
            break
print("저장한 학습 응답 수:", count)

## 4. 비교용 질문과 일반 디코딩 측정

평가 질문은 별도 `test_sft` split에서 가져온다. 같은 토큰 입력을 EAGLE-3 적용 후에도 사용한다. 2개 요청으로 워밍업한 뒤 평가 질문을 순차적으로 전송한다. 클라이언트에서 측정한 전체 응답시간과 출력 토큰 처리량이며, TTFT나 순수 GPU decode 시간은 아니다.

채택률은 vLLM의 draft·drafted token·accepted token 누적 counter 차이로 계산한다. 다른 클라이언트가 동시에 요청을 보내면 이 값이 섞이므로 측정 중에는 이 실습만 요청을 보낸다.

In [ ]:
eval_data = load_dataset(
    "HuggingFaceH4/ultrachat_200k", split="test_sft", streaming=True,
)
eval_prompts = []
for row in eval_data:
    prompt_ids = tokenizer.apply_chat_template(
        [row["messages"][0]], add_generation_prompt=True,
        enable_thinking=False, return_dict=False,
    )
    if len(prompt_ids) + MAX_NEW_TOKENS <= SEQ_LENGTH:
        eval_prompts.append(prompt_ids)
    if len(eval_prompts) == NUM_EVAL:
        break

def spec_counters():
    counters = {
        "vllm:spec_decode_num_drafts_total": 0,
        "vllm:spec_decode_num_draft_tokens_total": 0,
        "vllm:spec_decode_num_accepted_tokens_total": 0,
    }
    with urlopen(f"{BASE_URL}/metrics") as response:
        lines = response.read().decode().splitlines()
    for line in lines:
        if not line or line.startswith("#"):
            continue
        name = line.split("{", 1)[0].split()[0]
        if name in counters:
            counters[name] += float(line.split()[-1])
    return counters

def measure():
    for prompt in eval_prompts[:2]:
        complete(prompt)
    before = spec_counters()
    elapsed = []
    outputs = []
    for prompt in eval_prompts:
        start = perf_counter()
        outputs.append(complete(prompt))
        elapsed.append(perf_counter() - start)
    after = spec_counters()
    tokens = sum(len(ids) for ids in outputs)
    result = {
        "elapsed_s": sum(elapsed),
        "output_tokens": tokens,
        "output_tokens_per_s": tokens / sum(elapsed),
        "median_response_s": statistics.median(elapsed),
    }
    print(result)
    return result, outputs, {key: after[key] - before[key] for key in before}

baseline, baseline_tokens, _ = measure()

## 5. 전처리

Speculators의 공식 `prepare-data` 명령을 실행한다. 이미 토큰과 마스크가 있으므로 render endpoint를 사용하지 않는다. draft 어휘를 32,000개로 줄이기 위한 토큰 빈도도 계산한다. 전처리 단계에서 학습 가능한 응답 토큰이 없는 행은 제외될 수 있다.

In [ ]:
subprocess.run([
    *SPECULATORS, "prepare-data",
    "--model", MODEL,
    "--data", str(OUTPUT / "target_responses.jsonl"),
    "--output", str(OUTPUT / "prepared"),
    "--seq-length", str(SEQ_LENGTH),
    "--no-skip-token-freq",
], check=True)

## 6. Hidden state 추출 서버로 교체

일반 서버 터미널에서 **Ctrl+C로 종료**한 뒤, 아래 명령으로 추출 서버를 시작한다. 추출을 위해 chunked prefill을 끈다. 서버는 데이터의 전체 토큰을 입력으로 받고 해당 위치의 hidden state를 저장한다.

Qwen3.8-27B의 64개 층에서 보조 입력용 층 `[2, 32, 61]`과 target logit 계산용 마지막 층 `64`를 추출한다. 마지막 층은 출력 정규화 전 값이다. 학습 명령에는 보조 층 3개만 전달한다.

추출 서버에서는 성능을 측정하지 않는다. 파일 기록과 hidden state 수집 비용이 포함되기 때문이다.

In [ ]:
extraction_config = {
    "method": "extract_hidden_states",
    "num_speculative_tokens": 1,
    "draft_model_config": {
        "hf_config": {
            "eagle_aux_hidden_state_layer_ids": target_layers + [n_layers],
        },
    },
}
transfer_config = {
    "kv_connector": "ExampleHiddenStatesConnector",
    "kv_role": "kv_producer",
    "kv_connector_extra_config": {
        "shared_storage_path": str(OUTPUT / "transfer"),
    },
}
extraction_command = base_command + [
    "--disable-chunked-prefill",
    "--speculative-config", json.dumps(extraction_config),
    "--kv-transfer-config", json.dumps(transfer_config),
]
print(shlex.join(extraction_command))

서버 시작이 완료되면 아래 셀을 실행한다. 데이터 생성 명령은 완성된 전처리 행 수를 사용하며, 생성된 hidden state를 `hs_0.safetensors` 등의 데이터 행 순서로 저장한다. 숨은 상태의 마지막 차원은 5,120이고 추출 층 수는 4개다.

In [ ]:
subprocess.run([
    *SPECULATORS, "generate-offline-data",
    "--preprocessed-data", str(OUTPUT / "prepared"),
    "--endpoint", f"{BASE_URL}/v1",
    "--output", str(OUTPUT / "hidden_states"),
    "--concurrency", "4",
], check=True)

## 7. EAGLE-3 학습

추출이 끝나면 **서버를 Ctrl+C로 종료**한다. 오프라인 학습 중에는 vLLM 서버가 필요하지 않다.

학습은 Speculators의 EAGLE-3 기본 구조를 사용한다. target의 임베딩·출력 가중치와 저장된 hidden state를 이용하고 작은 draft decoder 등을 학습한다. draft 어휘 크기 32,000은 파라미터 수가 아니다. target 가중치는 갱신하지 않는다. Speculators가 전처리 데이터를 학습·검증으로 나누며, 앞의 `test_sft` 질문은 이 분할에 포함되지 않는다.

H100 프로필에서는 4개 GPU의 분산 학습, B300에서는 단일 GPU 학습을 실행한다. `train_eagle3.py`는 공식 학습 함수를 호출하며 실행 이력 저장만 제외한다. 학습 설정·모델 저장 방식은 README에 설명되어 있다.

In [ ]:
train_args = [
    "train_eagle3.py",
    "--verifier-name-or-path", MODEL,
    "--speculator-type", "eagle3",
    "--data-path", str(OUTPUT / "prepared"),
    "--hidden-states-path", str(OUTPUT / "hidden_states"),
    "--save-path", str(OUTPUT / "checkpoints"),
    "--draft-vocab-size", "32000",
    "--epochs", str(EPOCHS),
    "--total-seq-len", str(SEQ_LENGTH),
    "--target-layer-ids", *map(str, target_layers),
    "--on-missing", "raise",
]
train_command = [sys.executable, *train_args]
if TP > 1:
    train_command = [
        sys.executable, "-m", "torch.distributed.run",
        "--standalone", "--nproc_per_node", str(TP), *train_args,
    ]
train_env = os.environ.copy()
train_env["CUDA_VISIBLE_DEVICES"] = GPUS
subprocess.run(train_command, env=train_env, check=True)

학습 출력에서 train loss와 validation loss의 변화를 확인한다. 검증 손실이 개선될 때 Speculators가 `checkpoint_best` 링크를 갱신한다. 낮은 검증 손실이 곧 높은 실제 처리량을 보장하지는 않는다. 중단된 학습의 일부 출력만으로 완료 여부를 판단하지 않는다.

## 8. 학습한 EAGLE-3로 서버 시작

학습 프로세스가 끝난 뒤 아래 명령으로 서버를 시작한다. 동일한 target, BF16, GPU 수, 문맥 길이, 요청 상한을 유지하고 EAGLE-3만 추가한다. 한 번에 제안할 토큰 수는 3이다.

이 체크포인트는 학습에 사용한 Qwen3.8-27B와 함께 사용한다. 다른 Qwen 모델에 이름만 바꿔 연결하지 않는다.

In [ ]:
draft_config = {
    "method": "eagle3",
    "model": str(OUTPUT / "checkpoints" / "checkpoint_best"),
    "num_speculative_tokens": 3,
}
eagle_command = base_command + [
    "--speculative-config", json.dumps(draft_config),
]
print(shlex.join(eagle_command))

## 9. 적용 전후 비교

새 서버가 준비되면 아래 셀을 실행한다. 일반 서버에서 사용한 질문과 생성 조건을 그대로 사용한다.

- 가속비는 일반 실행의 총 응답시간을 EAGLE-3 실행의 총 응답시간으로 나눈 값이다.
- 채택률은 채택된 draft 토큰 수 / 제안된 draft 토큰 수다.
- draft 단계당 채택 길이는 채택된 draft 토큰 수 / draft 단계 수다. bonus token은 포함하지 않는다.
- greedy 출력 토큰 일치 여부도 확인한다. 표준 speculative decoding의 이론적 분포 보존과 GPU 연산의 비트 단위 동일성은 구분한다.

16개 질문은 동작 확인용이다. 실제 가속 여부는 대표 질문 수를 늘리고 동일 조건을 반복 측정해서 판단한다. 실습은 단일 동시 요청의 지연시간을 다루며 최대 서버 처리량을 측정하지 않는다.

In [ ]:
eagle, eagle_tokens, counters = measure()
drafts = counters["vllm:spec_decode_num_drafts_total"]
proposed = counters["vllm:spec_decode_num_draft_tokens_total"]
accepted = counters["vllm:spec_decode_num_accepted_tokens_total"]

print(f"총 응답시간 가속비: {baseline['elapsed_s'] / eagle['elapsed_s']:.3f}")
print("출력 토큰이 같은 질문 수:",
      sum(a == b for a, b in zip(baseline_tokens, eagle_tokens)),
      "/", len(eval_prompts))
assert drafts > 0 and proposed > 0, "EAGLE-3 draft counter를 확인하세요."
print(f"draft 토큰 채택률: {accepted / proposed:.3%}")
print(f"draft 단계당 채택 토큰 수: {accepted / drafts:.3f}")

## 10. 종료와 결과 해석

서버 터미널에서 Ctrl+C로 종료한다. 측정 결과는 노트북 셀 출력에 남는다. 저장된 결과만 읽을 때는 셀을 다시 실행하지 않는다.

EAGLE-3에는 draft 실행 비용과 검증 비용이 있다. 채택률이 낮거나 target 자체가 빠르면 가속비가 1보다 작을 수 있다. 학습 데이터와 평가 질문의 분포, 생성 모드, speculative token 수에 따라 결과가 달라진다. H100 4개와 B300 1개의 차이에는 GPU 수와 통신 비용도 포함된다.

Qwen3.8-27B는 hybrid attention 모델이다. 소스에서 EAGLE-3와 hidden state 수집 인터페이스를 확인했지만, 이 노트북을 해당 GPU에서 실측하지는 않았다. 설치 버전의 지원 여부와 학습 결과를 실제 실행으로 확인해야 한다.

## 참고 자료

- [vLLM speculative decoding](https://docs.vllm.ai/en/stable/features/speculative_decoding/)
- [Speculators: Train a Speculator](https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train/)
- [Speculators: Response Regeneration](https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/response_regeneration/)
- [Speculators 공식 오프라인 예제](https://github.com/vllm-project/speculators/blob/v0.9.0/examples/train/eagle3_llama3_8b_ultrachat_offline_5k.sh)
- [vLLM hidden state extraction](https://docs.vllm.ai/en/stable/features/speculative_decoding/extract_hidden_states/)
- [Qwen3.8-27B 모델 카드](https://huggingface.co/Qwen/Qwen3.8-27B)
- [UltraChat 데이터 카드](https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k)

vLLM·Speculators·Qwen3.8-27B는 Apache-2.0, UltraChat 데이터 카드는 MIT 라이선스를 명시한다. 학습 자료의 설명과 실행 코드는 이 실습에 맞게 작성했다.